0. IMPORT

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from pandas.core.reshape.merge import merge

print("Importing libraries completed successfully.")

Importing libraries completed successfully.


In [ ]:




print("Full Pipeline Complete! Columns ready for modeling:")
print(master.columns.tolist())

Full Pipeline Complete! Columns ready for modeling:
['year', 'country', 'iso3', 'region', 'coal_pct', 'oil_pct', 'gas_pct', 'nuclear_pct', 'hydro_pct', 'solar_pct', 'wind_pct', 'other_renewables_pct', 'renewables_total_pct', 'fossil_total_pct', 'co2_emissions_mt', 'population_millions', 'co2_per_capita_t', 'co2_intensity_kg_per_gdp_usd', 'market', 'temp_region', 'annual_avg_carbon_price', 'temp_anomaly_c', 'policy_event_count']


1. LOADING CSV FILES

In [3]:
energy_df = pd.read_csv('D:\\Bamm\\ICT Work\\Data Mining\\Data-Mining\\CSV\\carbon_prices_daily.csv')
co2_df = pd.read_csv('D:\\Bamm\\ICT Work\\Data Mining\\Data-Mining\\CSV\\co2_emissions_yearly.csv')
carbon_daily = pd.read_csv('D:\\Bamm\\ICT Work\\Data Mining\\Data-Mining\\CSV\\carbon_prices_daily.csv')
temp_monthly = pd.read_csv('D:\\Bamm\\ICT Work\\Data Mining\\Data-Mining\\CSV\\temperature_anomaly_monthly.csv')
events_data = pd.read_csv('D:\\Bamm\\ICT Work\\Data Mining\\Data-Mining\\CSV\\climate_events.csv')

print("Dataset Downloaded successfully.")

Dataset Downloaded successfully.


1.2. Clean column headers

In [2]:
for df in [energy_df, co2_df, carbon_daily, temp_monthly, events_data]:
    df.columns = df.columns.str.lower().str.strip()

1.3. Base Merge: Energy and CO2 using 'iso3'

In [3]:
co2_df = co2_df.drop(columns=['country', 'region'], errors='ignore')
master = energy_df.merge(co2_df, on=['iso3', 'year'], how='left')

1.4. Aggregate the Daily/Monthly datasets to Yearly

In [4]:
carbon_yearly = carbon_daily.groupby(['market', 'year'], as_index=False)['price'].mean()
carbon_yearly.rename(columns={'price': 'annual_avg_carbon_price'}, inplace=True)

temp_yearly = temp_monthly.groupby(['region', 'year'], as_index=False)['temp_anomaly_c'].mean()

temp_yearly.rename(columns={'region': 'temp_region'}, inplace=True) 

events_yearly = events_data.groupby(['region', 'year'], as_index=False)['is_policy'].sum()

events_yearly.rename(columns={'region': 'temp_region', 'is_policy': 'policy_event_count'}, inplace=True)

1.5. Mapping

In [ ]:
market_map = {
    'China': 'China_ETS', 'United Kingdom': 'UK_ETS', 'Germany': 'EU_ETS', 
    'France': 'EU_ETS', 'Italy': 'EU_ETS', 'Spain': 'EU_ETS', 'Poland': 'EU_ETS', 
    'Netherlands': 'EU_ETS', 'Czech Republic': 'EU_ETS', 'Belgium': 'EU_ETS', 
    'Sweden': 'EU_ETS', 'Finland': 'EU_ETS', 'Denmark': 'EU_ETS', 'Norway': 'EU_ETS', 
    'United States': 'California'
}

temp_map = {
    'United States': 'North_America', 'Canada': 'North_America', 'Mexico': 'North_America',
    'Germany': 'Europe', 'United Kingdom': 'Europe', 'France': 'Europe', 'Italy': 'Europe',
    'Spain': 'Europe', 'Poland': 'Europe', 'Netherlands': 'Europe', 'Ukraine': 'Europe',
    'Czech Republic': 'Europe', 'Belgium': 'Europe', 'Sweden': 'Europe', 'Norway': 'Europe',
    'Finland': 'Europe', 'Denmark': 'Europe', 'Russia': 'Europe',
    'China': 'Asia', 'India': 'Asia', 'Japan': 'Asia', 'Iran': 'Asia', 'South Korea': 'Asia',
    'Indonesia': 'Asia', 'Saudi Arabia': 'Asia', 'Turkey': 'Asia', 'UAE': 'Asia', 'Iraq': 'Asia',
    'Qatar': 'Asia', 'Kuwait': 'Asia', 'Israel': 'Asia', 'Vietnam': 'Asia', 'Thailand': 'Asia',
    'Pakistan': 'Asia', 'Malaysia': 'Asia', 'Philippines': 'Asia', 'Bangladesh': 'Asia', 'Kazakhstan': 'Asia',
    'Brazil': 'Tropics', 'Colombia': 'Tropics', 'Venezuela': 'Tropics', 'Nigeria': 'Tropics', 'Kenya': 'Tropics',
    'South Africa': 'Southern_Hemisphere', 'Australia': 'Southern_Hemisphere', 'Argentina': 'Southern_Hemisphere', 'Chile': 'Southern_Hemisphere',
    'Egypt': 'Northern_Hemisphere', 'Algeria': 'Northern_Hemisphere', 'Morocco': 'Northern_Hemisphere'
}

1.6. Apply Mappings to Master

In [ ]:
master['market'] = master['country'].map(market_map)
master['temp_region'] = master['country'].map(temp_map)

1.7. Final Merges

In [ ]:
master = master.merge(carbon_yearly, on=['market', 'year'], how='left')
master = master.merge(temp_yearly, on=['temp_region', 'year'], how='left')
master = master.merge(events_yearly, on=['temp_region', 'year'], how='left')